# CIC6314 — Smart Product Recommendation System
**Integration Layer** | **Dataset:** UCI Online Retail (Dec 2010 – Dec 2011)

**Pipeline:** Rules Engine → A\* Search → RF Blend + ALS → Output

---

This notebook is the integration layer of our Smart Product Recommendation System. Its job is to tie together the three upstream components — the rules engine (Member 2), the A\* search module (Member 1), and the ML model (Member 3) — into a single callable function `recommend()` that takes a customer profile and returns a list of product recommendations.

There are two different recommendation paths depending on whether the customer has purchase history or not:

- **Personalised path** — for returning customers who have bought something before. Their favourite category is used as the starting node for A\* search, which narrows down the eligible categories. Then the Random Forest model scores those categories and the top 3 are passed to ALS to pick the actual products.

- **Cold-start path** — for new customers with no purchase history. We can't run A\* or the history model because there's no starting point and no data to work with. Instead we just return the most widely purchased products in their eligible categories.

The dataset is the UCI Online Retail dataset — 397,884 real transactions from a UK gift and novelty retailer between December 2010 and December 2011. It covers 4,338 customers, 3,665 products, and 8 product categories.

## 1. Imports & Artefact Loading

We need four pre-trained model files that were saved by Member 3's `ml_model.ipynb`. These are all pickle (`.pkl`) files stored in the `models/` directory:

- **`similarity_matrix.pkl`** — a 3,665 × 3,665 item-item cosine similarity matrix produced by ALS (Alternating Least Squares). Each cell `[i][j]` is the cosine similarity between product i and product j in the latent factor space. Used by `recommend_products()` to find products similar to what the customer has already bought.

- **`product_catalogue.pkl`** — a DataFrame containing every product's StockCode, description, category, average unit price, and popularity rank (unique buyer count). Used for both the personalised and cold-start paths.

- **`model_context.pkl`** — the context Random Forest model (Model A). Trained on 3 features: customer segment, price range, and current month. Works for all customers including new ones because it doesn't need purchase history.

- **`model_history.pkl`** — the history Random Forest model (Model B). Trained on 15 features including number of purchases, number of unique categories, recency, average order value, and a one-hot encoded favourite category. Only used for returning customers who have purchase history.

> **Important:** The `segment_map`, `price_map`, and `all_fav_cats` encoding constants defined below must match exactly what was used during training in `ml_model.ipynb`. If these are changed, the model outputs will be wrong because the feature vectors will no longer align with what the model was trained on.

In [57]:
import pickle
import numpy as np
import pandas as pd
import sys
import os
import warnings
warnings.filterwarnings('ignore')

# ── path setup: works whether run from /notebooks or repo root ────────────────
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

# ── shared modules ────────────────────────────────────────────────────────────
from src.constants    import PRODUCT_CATEGORIES, SAMPLE_PROFILES, build_user_profile
from src.rules_engine import apply_rules
from src.search_module import find_reachable_categories, find_popular_categories

# ── artefact loading ──────────────────────────────────────────────────────────
print('Loading artefacts...')
item_sim_df       = pickle.load(open('models/similarity_matrix.pkl',  'rb'))
product_catalogue = pickle.load(open('models/product_catalogue.pkl',  'rb'))
model_context     = pickle.load(open('models/model_context.pkl',      'rb'))
model_history     = pickle.load(open('models/model_history.pkl',      'rb'))
print('All artefacts loaded.')
print(f'  similarity_matrix : {item_sim_df.shape}')
print(f'  product_catalogue : {product_catalogue.shape}')
print(f'  model_context     : MultiOutputClassifier ({len(model_context.estimators_)} estimators, 3 features)')
print(f'  model_history     : MultiOutputClassifier ({len(model_history.estimators_)} estimators, 15 features)')

# ── encoding constants (MUST match ml_model.ipynb training) ──────────────────
# Estimators in both models are ordered by PRODUCT_CATEGORIES (the label column order used during training).
# all_fav_cats must match the one-hot encoding used to build the 15-feature history vector.
segment_map  = {'New': 0, 'Occasional': 1, 'Frequent': 2}
price_map    = {'Low': 0, 'Mid-Low': 1, 'Mid-High': 2, 'High': 3}
all_fav_cats = PRODUCT_CATEGORIES + ['Unknown']   # 9 items: 8 categories + Unknown fallback

Loading artefacts...
All artefacts loaded.
  similarity_matrix : (3665, 3665)
  product_catalogue : (3897, 5)
  model_context     : MultiOutputClassifier (8 estimators, 3 features)
  model_history     : MultiOutputClassifier (8 estimators, 15 features)


## 2. Inference Functions (from ml_model.ipynb Section 9)

These three functions were written by **Member 3** and are copied here from `ml_model.ipynb` so this notebook can run on its own without needing to re-run the training notebook. We do not modify them — the feature ordering and encoding must stay identical to what the models were trained on.

### The Three Functions

**`extract_context_features(user_profile)`** — builds the 3-feature input vector for Model A. The three features are customer segment (encoded as 0/1/2), price range (encoded as 0/1/2/3), and the current calendar month. These features are always available regardless of purchase history.

**`extract_history_features(user_profile)`** — builds the 15-feature input vector for Model B. The extra features on top of Model A are: number of items purchased, number of unique categories visited, days since last order, average order value, and a 9-element one-hot vector for favourite category (8 categories + Unknown).

**`predict_product(user_profile, candidates)`** — the main scoring function. It runs both models and blends their outputs using a confidence weight based on how much purchase history the customer has.

### How the confidence blend works

The blend formula is:

```
confidence = 1 - 1 / (1 + n_purchases)
final_score = (1 - confidence) * context_score + confidence * history_score
```

This means:

```
n = 0   purchases  → confidence = 0.00  → 100% context model,  0% history model
n = 1   purchase   → confidence = 0.50  →  50% context model, 50% history model
n = 5   purchases  → confidence = 0.83  →  17% context model, 83% history model
n = 10  purchases  → confidence = 0.91  →   9% context model, 91% history model
n = 14  purchases  → confidence = 0.93  →   7% context model, 93% history model
```

The idea is that for a brand new customer (n=0), we have no purchase signal at all so we rely entirely on the context model which just uses their spend tier and the current month. As they buy more things, we trust the history model more because it has real data to work with. By the time a customer has 10+ purchases, the history model is doing almost all the work.

**`recommend_products(user_profile, category, top_n)`** — given a category, finds the top N products within it that are most similar to what the customer has already bought. Similarity is computed as the mean ALS cosine similarity between each candidate product and all items in the customer's purchase history. If the customer has no history (or none of their products are in the similarity matrix), it falls back to ranking by popularity instead.

In [58]:
def extract_context_features(user_profile):
    """
    Build the 3-feature context vector used by Model A.
    Features: customer_segment (encoded), price_range (encoded), current month.
    Always available — works for new and returning customers alike.
    """
    return [[
        segment_map.get(user_profile['customer_segment'], 0),
        price_map.get(user_profile['price_range'], 0),
        pd.Timestamp.now().month
    ]]


def extract_history_features(user_profile):
    """
    Build the 15-feature history vector used by Model B.
    Features: n_purchases, n_categories, recency_days, avg_order_value,
              segment_enc, price_enc, favourite_category (one-hot × 9).
    Defaults to zero for all history fields when purchase_history is empty.
    """
    fav = user_profile.get('favourite_category') or 'Unknown'
    if fav not in all_fav_cats:
        fav = 'Unknown'
    fav_ohe = [1 if c == fav else 0 for c in all_fav_cats]   # 9-element one-hot
    return [[
        len(user_profile['purchase_history']),
        len(user_profile.get('purchased_categories', [])),
        user_profile.get('recency_days', 999),
        user_profile.get('avg_order_value', 0.0),
        segment_map.get(user_profile['customer_segment'], 0),
        price_map.get(user_profile['price_range'], 0),
        *fav_ohe
    ]]


def predict_product(user_profile, candidates=None):
    """
    Score candidate categories using a confidence-blended RF model.
    Works for ALL users — new users get 100% context model; returning users blend both.

    Parameters
    ----------
    user_profile : dict   from build_user_profile()
    candidates   : list   subset of PRODUCT_CATEGORIES to score (default: all 8)

    Returns
    -------
    list[tuple[str, float]]  [(category, probability), ...] sorted descending
    """
    cats       = candidates or PRODUCT_CATEGORIES
    n_hist     = len(user_profile['purchase_history'])
    confidence = 1.0 - 1.0 / (1.0 + n_hist)

    # Model A — context scores (always computed)
    X_ctx      = extract_context_features(user_profile)
    ctx_scores = np.array([
        est.predict_proba(X_ctx)[0][1] if len(est.classes_) > 1 else 0.0
        for est in model_context.estimators_
    ])

    # Model B — history scores (only meaningful for returning users)
    if n_hist > 0:
        X_hist      = extract_history_features(user_profile)
        hist_scores = np.array([
            est.predict_proba(X_hist)[0][1] if len(est.classes_) > 1 else 0.0
            for est in model_history.estimators_
        ])
    else:
        hist_scores = np.zeros(len(PRODUCT_CATEGORIES))

    # Blend by confidence — estimators are in PRODUCT_CATEGORIES order
    blended = (1.0 - confidence) * ctx_scores + confidence * hist_scores
    scores  = {cat: float(blended[i])
               for i, cat in enumerate(PRODUCT_CATEGORIES) if cat in cats}
    return sorted(scores.items(), key=lambda x: x[1], reverse=True)


def recommend_products(user_profile, category, top_n=3):
    """
    Recommend specific products within a category using ALS item similarity.
    Finds products most similar to the customer's purchase history.
    Falls back to global popularity if the customer has no purchase history.

    Parameters
    ----------
    user_profile : dict   from build_user_profile()
    category     : str    one of PRODUCT_CATEGORIES
    top_n        : int    number of products to return (default 3)

    Returns
    -------
    list[dict]  [{'category', 'product', 'price', 'score'}, ...]
                score is cosine similarity float, or 0.0 if using popularity fallback
    """
    bought       = user_profile['purchase_history']
    valid_bought = [sc for sc in bought if sc in item_sim_df.columns]

    cat_items = product_catalogue[product_catalogue['category'] == category].copy()
    unowned   = cat_items[~cat_items['StockCode'].isin(bought)]

    # Fallback: no purchase history or no unowned items in category → popularity
    if not valid_bought or unowned.empty:
        top = unowned.nlargest(top_n, 'popularity_rank')
        return [{'category': category,
                 'product':  row['Description'],
                 'price':    round(float(row['avg_price']), 2),
                 'score':    0.0}
                for _, row in top.iterrows()]

    # ALS similarity: mean similarity of candidate products to all owned products
    valid_unowned = unowned[unowned['StockCode'].isin(item_sim_df.index)].copy()
    if valid_unowned.empty:
        top = unowned.nlargest(top_n, 'popularity_rank')
        return [{'category': category,
                 'product':  row['Description'],
                 'price':    round(float(row['avg_price']), 2),
                 'score':    0.0}
                for _, row in top.iterrows()]

    valid_unowned['score'] = (
        item_sim_df.loc[valid_unowned['StockCode'], valid_bought]
                   .mean(axis=1).values
    )
    top = valid_unowned.nlargest(top_n, 'score')
    return [{'category': category,
             'product':  row['Description'],
             'price':    round(float(row['avg_price']), 2),
             'score':    round(float(row['score']), 4)}
            for _, row in top.iterrows()]


print('Inference functions ready: predict_product(), recommend_products()')

Inference functions ready: predict_product(), recommend_products()


## 3. `recommend()` — Main Orchestrator

`recommend()` is the only function that the end user (or any external system) needs to call. It takes a customer profile dictionary and returns a standardised output dictionary. Everything else — rules, search, ML scoring, product lookup — happens inside.

### Routing Logic

The function checks `purchase_history` first. If it's empty, the customer is new and we go down the cold-start path. If they have history, we go down the personalised path.

```
purchase_history = []          → COLD-START path
  apply_rules() → find_popular_categories() → get_popular_products()
  recommendation_type = 'popular'
  scores = int (unique buyer count)

purchase_history = ['85123A', ...]  → PERSONALISED path
  apply_rules() → find_reachable_categories() → predict_product() → recommend_products()
  recommendation_type = 'personalised'
  scores = float (RF probability 0–1)
```

### Personalised Path — Step by Step

1. **Rules engine** (`apply_rules`) — checks the customer's spend tier and buying behaviour to decide which product categories they're allowed to see. For example, a Low spend customer starts with only Home Decor, Stationery & Craft, and Seasonal & Gifts. Rules 5–9 can open up more categories based on their history.

2. **A\* search** (`find_reachable_categories`) — takes the eligible categories and the customer's favourite category, and does A\* search on the 8-node ALS similarity graph to find which eligible categories are reachable within 2 hops. The results are ordered by A\* f-score (lower f = more optimal) so the ML model gets the most relevant candidates first.

3. **RF scoring** (`predict_product`) — scores every category in the A\* shortlist using the confidence-blended Random Forest. Returns them ranked by predicted purchase probability.

4. **Product lookup** (`recommend_products`) — for each of the top 3 categories, finds the 3 most similar products to the customer's purchase history using ALS item similarity. Returns 9 products total.

### Cold-Start Path — Step by Step

1. **Rules engine** (`apply_rules`) — same as above. New customers hit Rule 10 which restricts them to Home Decor, Seasonal & Gifts, and Kitchen & Dining.

2. **Popular categories** (`find_popular_categories`) — picks the top 3 eligible categories by total unique buyer count from the product catalogue.

3. **Popular products** (`get_popular_products`) — for each category, returns the 3 most widely purchased products by buyer count. No personalisation — this is just the global baseline.

### Output Schema

```python
{
    'recommendation_type':  'personalised' | 'popular',
    'top_3_categories':     [(category, score), ...]         # exactly 3 items
    'recommended_products': [{category, product, price, score}, ...]  # exactly 9 items
    'eligible':             [category, ...]                  # from apply_rules()
}
```

The `score` field means different things on each path — RF probability (float) for personalised, unique buyer count (int) for popular. The `display_recommendation()` function uses `isinstance(score, float)` to detect which one it got and format accordingly.

In [59]:
def recommend(user_profile: dict) -> dict:
    """
    Main recommendation entry point. Routes between personalised and cold-start paths.

    Parameters
    ----------
    user_profile : dict   built by build_user_profile() from src.constants

    Returns
    -------
    dict with keys:
        recommendation_type  : 'personalised' | 'popular'
        top_3_categories     : [(category, score), ...]   — 3 items
        recommended_products : [{category, product, price, score}, ...] — 9 items
        eligible             : [category, ...]            — from apply_rules()
    """
    # Step 1: Rules engine — gate eligible categories
    eligible = apply_rules(user_profile)

    # ── COLD-START PATH ───────────────────────────────────────────────────────
    # No purchase history → ML model and BFS cannot personalise.
    # Fall back to globally popular categories and products within the eligible set.
    if not user_profile['purchase_history']:
        categories = find_popular_categories(
            eligible, user_profile['price_range'], top_n=3
        )
        products = []
        for cat, _ in categories:
            products += get_popular_products(cat, top_n=3)

        return {
            'recommendation_type':  'popular',
            'top_3_categories':     categories,
            'recommended_products': products,
            'eligible':             eligible,
        }

    # ── PERSONALISED PATH ─────────────────────────────────────────────────────
    # Step 2: BFS search — narrow eligible to categories reachable from favourite
    shortlist = find_reachable_categories(user_profile, eligible)

    # Step 3: RF blend — score and rank the shortlisted categories
    categories = predict_product(user_profile, candidates=shortlist)[:3]

    # Step 4: ALS item similarity — recommend 3 products per top category
    products = []
    for cat, _ in categories:
        products += recommend_products(user_profile, category=cat, top_n=3)

    return {
        'recommendation_type':  'personalised',
        'top_3_categories':     categories,
        'recommended_products': products,
        'eligible':             eligible,
    }


print('recommend() ready.')

recommend() ready.


## 4. apply_rules() function. Knowledge Representation & Logical Reasoning using Rules Engine 

### Module Overview & Functional Role
The Rules Engine (`src/rules_engine.py`) is the structural entry point of the system pipeline. It has the main goal of handling the "Cold-Start Problem" for brand-new customers, as well as setting initial candidate ranges for customers who return by using behavioral signals to map the customer to an eligible subset of the 8 core product categories. 
Running the deterministic heuristic filter prior to the execution of heavier machine learning models lowers the computational complexity of these systems, sets the boundaries of risk in recommendations, and offers a structural fail-safe fallback.

### The 10 Production Rules & Business Rationale
The module processes a customer's profile using 10 well-defined rules:

* **Spend-Tier Filtering (Rules 1–4):** Restricts or expands available inventory vectors depending on the customer's calculated financial profile (`price_range`) to ensure that the right amount of exposure is being given for their budget.
* **Core Retention Loop (Rule 5):** Ensures a returning customer's historical `favourite_category` value is never lost because of down-stream processing nodes.
* **Exploration vs. Specialization Controls (Rules 6–7):** Shoppers who purchase across 3 categories can access the entire catalog. Additionally, if an individual shopper has only one category purchased are gently suggested a single, optimal neighboring category calculated using an Alternating Least Squares (ALS) similarity matrix (`models/category_similarity.pkl`) to avoid choice overload.
* **Churn Countermeasure (Rule 8):** Customers who are not active for $>90$ days are given very high levels of impulse triggers, such as `Seasonal & Gifts`, `Food & Confectionery` to re-engage.
* **Power Shoppers (Rule 9):** Disables all filtering of the high-frequency 'Frequent' segments.
* **Cold-Start Absolute Override (Rule 10):** Forces other rule mutations out of the picture if a customer is flagged as `New` and funnels them exclusively into the top 3 highest-confidence popularity baseline categories.

### Technical Specifications & System Constraints
Due to the risk of the pipeline crashing at runtime because of architectural issues, the implementation makes careful use of structural data bounds:
* **Deterministic Sorting:** The final category strings are filtered through a list comprehension map tracking `PRODUCT_CATEGORIES` to bypass non-deterministic runtime set-hashing orders. This provides predictable arrays to the downstream Search module.
* **Fail-Safe Fallback:** If anomalous behavioral vectors result in a null candidate set, an automated catch resets the output to the full catalogue array as an absolute pipeline fail-safe.

### Execution Mechanics: When is `apply_rules()` Called?

The function `apply_rules(user_profile)` is called at the start of the recommendation lifecycle immediately after a user_profile dictionary is compiled. 

#### Returning Users Workflow
For returning customers with established purchase histories, the rules engine sets the maximum ceiling of possibilities. Its filtered output array is directly fed as the candidate shortlist constraint into Search Module and Machine Learning Model:
$$\text{User Profile} \longrightarrow \textbf{Rules Engine (Eligible Shortlist)} \longrightarrow \text{Search Module (Graph Reachability)} \longrightarrow \text{ML Classifier (Personalized Scoring)}$$

#### New Users Workflow 
If a user is verified as a completely new customer with an empty purchase history, the integration layer routes the output of the Rules Engine directly to a popularity baseline module, completely bypassing graph traversal and machine learning classification.

## 5. `get_popular_products()` — Cold-Start Helper

This function is only used on the cold-start path. When a customer has no purchase history we can't compute ALS similarity scores (there's nothing to compare against), so instead we just return the products that the most people have bought in that category.

The `popularity_rank` column in the product catalogue is the number of unique customers who bought each product across all transactions in the dataset. So a score of 2,000 means 2,000 different customers bought that product at least once.

One important detail: this function returns `score` as a Python `int`, not a `float`. That's deliberate — `display_recommendation()` checks `isinstance(score, float)` to decide how to format the score label. If it's a float it shows it as a probability, if it's an int it shows it as `X buyers`. So the type itself carries information about which path was used.

In [60]:
def get_popular_products(category: str, top_n: int = 3) -> list:
    """
    Return the most globally purchased products in a given category.
    Used on the cold-start path when no purchase history is available.

    Parameters
    ----------
    category : str   one of PRODUCT_CATEGORIES
    top_n    : int   number of products to return (default 3)

    Returns
    -------
    list[dict]  [{'category', 'product', 'price', 'score'}, ...]
                score is int (unique buyer count) — NOT a float probability
    """
    cat_items = product_catalogue[product_catalogue['category'] == category]
    top       = cat_items.nlargest(top_n, 'popularity_rank')
    return [
        {
            'category': category,
            'product':  row['Description'],
            'price':    round(float(row['avg_price']), 2),
            'score':    int(row['popularity_rank']),  # int, not float
        }
        for _, row in top.iterrows()
    ]


print('get_popular_products() ready.')

get_popular_products() ready.


## 6. `display_recommendation()` — Output Formatter

This is just a display utility — it doesn't change any data, it just prints the output of `recommend()` in a readable format.

The main challenge here is that the two recommendation paths return scores in different formats. The personalised path gives float probabilities between 0 and 1, while the cold-start path gives integer buyer counts that can be in the hundreds of thousands. We can't use the same format string for both.

The solution is to check `isinstance(score, float)`. If True, format as a 4-decimal probability. If False (it's an int), format as a comma-separated number with `buyers` appended. This works because `get_popular_products()` explicitly casts scores to Python `int` — not numpy int64, but actual `int` — so the isinstance check is reliable.

The function also groups products by category in the output, using a `prev_cat` tracker so it only prints the category header when the category changes.

In [61]:
def display_recommendation(result: dict, customer_name: str = '') -> None:
    """
    Pretty-print a recommend() output dict.
    Handles float (personalised) and int (popular) score types.

    Parameters
    ----------
    result        : dict   output of recommend()
    customer_name : str    optional label for display
    """
    is_personalised = result['recommendation_type'] == 'personalised'
    tag = 'Recommended for you' if is_personalised else 'Popular right now'

    print(f"\n{'═'*65}")
    if customer_name:
        print(f"  Customer   : {customer_name}")
    print(f"  Mode       : {result['recommendation_type'].upper()}  ({tag})")
    print(f"{'─'*65}")
    print(f"  Eligible   : {result['eligible']}")

    print(f"\n  Top 3 Categories:")
    for rank, (cat, score) in enumerate(result['top_3_categories'], 1):
        label = f"{score:.4f}" if isinstance(score, float) else f"{score:,} buyers"
        print(f"    {rank}. {cat:<30} {label}")

    print(f"\n  Recommended Products:")
    prev_cat = None
    for item in result['recommended_products']:
        if item['category'] != prev_cat:
            print(f"    ── {item['category']} ──")
            prev_cat = item['category']
        label = f"{item['score']:.4f}" if isinstance(item['score'], float) \
                else f"{item['score']:,} buyers"
        print(f"      {item['product'][:45]:<45} £{item['price']:>6.2f}  {label}")
    print(f"{'═'*65}")


print('display_recommendation() ready.')

display_recommendation() ready.


## 6. Demo: All 5 Sample Profiles

We run the full pipeline on all five sample profiles defined in `src/constants.py`. These profiles are based on real CustomerIDs from the UCI Online Retail dataset including their purchase histories, average order values, and favourite categories all come from actual transaction data.

| Profile | Customer ID | Segment | Spend Tier | Favourite Category | Expected Path |
|---|---|---|---|---|---|
| `gift_buyer` | 13058 | Occasional | Low | Seasonal & Gifts | personalised |
| `home_decorator` | 13094 | Frequent | Low | Home Decor | personalised |
| `kitchen_enthusiast` | 13631 | Frequent | Mid-Low | Kitchen & Dining | personalised |
| `craft_lover` | 14460 | Occasional | Low | Stationery & Craft | personalised |
| `new_customer` | NEW_001 | New | Low | None | **popular (cold-start)** |

All four returning customers should go through the personalised path. The new_customer has no purchase history, so it goes through cold-start. Rule 10 in the rules engine overrides everything for New segment customers and restricts them to only 3 categories: Home Decor, Seasonal & Gifts, and Kitchen & Dining.

In [62]:
results = {}
for name, profile in SAMPLE_PROFILES.items():
    results[name] = recommend(profile)
    display_recommendation(results[name], customer_name=name)


═════════════════════════════════════════════════════════════════
  Customer   : gift_buyer
  Mode       : PERSONALISED  (Recommended for you)
─────────────────────────────────────────────────────────────────
  Eligible   : ['Home Decor', 'Kitchen & Dining', 'Seasonal & Gifts', 'Toys & Games', 'Stationery & Craft', 'Fashion & Accessories', 'Garden & Outdoor', 'Food & Confectionery']

  Top 3 Categories:
    1. Home Decor                     0.5641
    2. Seasonal & Gifts               0.4972
    3. Stationery & Craft             0.2465

  Recommended Products:
    ── Home Decor ──
      PINK BABY BUNTING                             £  2.97  0.5552
      PARTY BUNTING                                 £  4.88  0.5206
      VINTAGE UNION JACK BUNTING                    £  8.51  0.4652
    ── Seasonal & Gifts ──
      WOODEN HAPPY BIRTHDAY GARLAND                 £  2.97  0.3696
      GARLAND WOODEN HAPPY EASTER                   £  1.25  0.2736
      VINTAGE CHRISTMAS STOCKING            

### Verification Checklist

These automated checks verify that the output of `recommend()` matches the expected schema for all five profiles. We check 8 things per profile:

1. The output dictionary has exactly the 4 expected keys
2. `recommendation_type` is either `'personalised'` or `'popular'`
3. New customers (no purchase history) always get `'popular'`
4. Returning customers (with purchase history) always get `'personalised'`
5. `top_3_categories` always has exactly 3 items
6. `recommended_products` always has exactly 9 items (3 per category)
7. All product dicts have the 4 required keys: `category`, `product`, `price`, `score`
8. Score types match the path — float for personalised, int for popular

In [63]:
PASS = '\033[92m PASS \033[0m'
FAIL = '\033[91m FAIL \033[0m'

def check(label, condition):
    print(f'  [{PASS if condition else FAIL}] {label}')
    return condition

all_passed = True
print('\n── Schema & Routing Checks ──')
for name, result in results.items():
    profile = SAMPLE_PROFILES[name]
    print(f'\n{name}:')
    ok = True
    ok &= check('Has exactly 4 keys',
                set(result.keys()) == {'recommendation_type','top_3_categories','recommended_products','eligible'})
    ok &= check('recommendation_type is valid string',
                result['recommendation_type'] in ('personalised','popular'))
    ok &= check('Cold-start → popular',
                not (not profile['purchase_history'] and result['recommendation_type'] != 'popular'))
    ok &= check('Returning → personalised',
                not (profile['purchase_history'] and result['recommendation_type'] != 'personalised'))
    ok &= check('top_3_categories has exactly 3 items',
                len(result['top_3_categories']) == 3)
    ok &= check('recommended_products has exactly 9 items',
                len(result['recommended_products']) == 9)
    ok &= check('All product dicts have required keys',
                all({'category','product','price','score'} <= set(p.keys())
                    for p in result['recommended_products']))
    if result['recommendation_type'] == 'personalised':
        ok &= check('Personalised scores are float',
                    all(isinstance(p['score'], float) for p in result['recommended_products']))
    else:
        ok &= check('Popular scores are int',
                    all(type(p['score']) is int for p in result['recommended_products']))
    all_passed = all_passed and ok

print(f'\n{"All checks passed ✓" if all_passed else "Some checks FAILED — review above"}')


── Schema & Routing Checks ──

gift_buyer:
  [ PASS ] Has exactly 4 keys
  [ PASS ] recommendation_type is valid string
  [ PASS ] Cold-start → popular
  [ PASS ] Returning → personalised
  [ PASS ] top_3_categories has exactly 3 items
  [ PASS ] recommended_products has exactly 9 items
  [ PASS ] All product dicts have required keys
  [ PASS ] Personalised scores are float

home_decorator:
  [ PASS ] Has exactly 4 keys
  [ PASS ] recommendation_type is valid string
  [ PASS ] Cold-start → popular
  [ PASS ] Returning → personalised
  [ PASS ] top_3_categories has exactly 3 items
  [ PASS ] recommended_products has exactly 9 items
  [ PASS ] All product dicts have required keys
  [ PASS ] Personalised scores are float

kitchen_enthusiast:
  [ PASS ] Has exactly 4 keys
  [ PASS ] recommendation_type is valid string
  [ PASS ] Cold-start → popular
  [ PASS ] Returning → personalised
  [ PASS ] top_3_categories has exactly 3 items
  [ PASS ] recommended_products has exactly 9 items
  [ P

### Pipeline Trace: Watching a Recommendation Being Built

This section traces through a single recommendation step by step for the `home_decorator` profile. The goal is to make the internal reasoning of the pipeline visible of what each module receives, what it returns, and how the output changes at each stage.

**Profile summary:** Customer 13094 is a Frequent buyer with Low spend (avg order value £80.31). Their favourite category is Home Decor and they've bought products from Home Decor and Food & Confectionery. They were last active 20 days ago.

In [64]:
profile = SAMPLE_PROFILES['home_decorator']
print('── Pipeline Trace: home_decorator ──')
print(f'  purchase_history   : {profile["purchase_history"]}')
print(f'  favourite_category : {profile["favourite_category"]}')
print(f'  customer_segment   : {profile["customer_segment"]}')
print(f'  price_range        : {profile["price_range"]}')
print()

eligible  = apply_rules(profile)
print(f'Step 1 — apply_rules()         : {eligible}')

shortlist = find_reachable_categories(profile, eligible)
print(f'Step 2 — find_reachable_cats() : {shortlist}')
pruned    = [c for c in eligible if c not in shortlist]
if pruned:
    print(f'         (pruned by A*)         : {pruned}')

ranked    = predict_product(profile, candidates=shortlist)
print(f'Step 3 — predict_product()     :')
for cat, score in ranked:
    marker = '  <- top 3' if (cat, score) in ranked[:3] else ''
    print(f'           {cat:<30} {score:.4f}{marker}')

top3 = ranked[:3]
print(f'Step 4 — recommend_products() x 3 categories:')
for cat, _ in top3:
    prods = recommend_products(profile, category=cat, top_n=3)
    print(f'  [{cat}]')
    for p in prods:
        print(f'    {p["product"][:45]:<45} £{p["price"]:>6.2f}  sim={p["score"]:.4f}')


── Pipeline Trace: home_decorator ──
  purchase_history   : ['22174', '22791', '84946', '85123A']
  favourite_category : Home Decor
  customer_segment   : Frequent
  price_range        : Low

Step 1 — apply_rules()         : ['Home Decor', 'Kitchen & Dining', 'Seasonal & Gifts', 'Toys & Games', 'Stationery & Craft', 'Fashion & Accessories', 'Garden & Outdoor', 'Food & Confectionery']
Step 2 — find_reachable_cats() : ['Home Decor', 'Food & Confectionery', 'Stationery & Craft', 'Garden & Outdoor', 'Kitchen & Dining', 'Toys & Games', 'Seasonal & Gifts']
         (pruned by A*)         : ['Fashion & Accessories']
Step 3 — predict_product()     :
           Home Decor                     0.8544  <- top 3
           Kitchen & Dining               0.3163  <- top 3
           Toys & Games                   0.2925  <- top 3
           Seasonal & Gifts               0.2768
           Garden & Outdoor               0.2708
           Stationery & Craft             0.2482
           Food & Confecti

## 7. Results & Discussion

### Why Two Paths?

We designed the system around two distinct paths because the available data is fundamentally different between a new customer and a returning one. For a new customer with no purchase history, running A\* search makes no sense as there's no favourite category to use as a starting node. Running the history RF model also makes no sense as all 15 history features would be zero or unknown. So rather than trying to force a personalised recommendation with no data behind it, we fall back to the simplest reliable thing: show the most popular products in their eligible categories.

For returning customers, the personalised path is much more meaningful. The customer's purchase history anchors the A\* search, the history model picks up category preferences from their buying pattern, and the ALS similarity matrix finds products that are similar to what they've already bought.

### Score Types

The two paths return different score types and that's by design:

**Personalised path scores** are Random Forest predicted probabilities, which is the values between 0 and 1 representing how likely the model thinks this customer is to buy from a given category. For example, a score of 0.73 means the blended RF model predicts a 73% chance the customer will purchase from that category. These are meaningful relative to each other. With a score of 0.73 should be ranked above 0.61.

**Cold-start path scores** are raw unique buyer counts from the dataset — the number of distinct customers who bought at least one product in that category. These are not probabilities; they're popularity signals. A score of 142,309 for Home Decor just means 142,309 unique customers bought something from Home Decor across the entire dataset period. They're meaningful relative to each other (Home Decor is more popular than Garden & Outdoor) but they're not in the same space as the personalised scores.

We use `isinstance(score, float)` as the discriminator in `display_recommendation()` because `get_popular_products()` explicitly returns Python `int`, not float.

### What We Observed in the Demo

Home Decor ranked first for all four returning customers, even for customers whose favourite category is something else like Kitchen & Dining or Stationery & Craft. This is because Home Decor has by far the highest buyer count in the dataset (142,309 unique buyers), so the RF model developed a strong prior toward it during training. The history model does pick up individual preference where the customer's actual favourite category usually shows up second or third, but it's not strong enough to override the Home Decor prior in most cases.

At the product level the recommendations are more individualised. The `craft_lover` profile gets Stationery & Craft products with high ALS similarity to their 14 purchased items. The `kitchen_enthusiast` gets Kitchen & Dining products (including Regency-brand items) that match the Regency kitchenware already in their history. The ALS item similarity is doing real work at this level even if the category ranking is biased toward popularity.

### Known Limitations

**Home Decor dominance at the category level.** The RF models reflect the training data distribution. Home Decor appears in far more training examples than any other category, so the model learned to predict it most often. A fix would be to apply class weighting or inverse frequency weighting during training, but we did not implement this.

**Sparse ALS matrix.** The item-item similarity matrix is based on a user-item matrix that is only 1.54% dense. Many product pairs have never been co-purchased by the same customer, so their ALS similarity is zero or near-zero. When a customer's purchased items have no similarity signal to candidates in a category, `recommend_products()` falls back to popularity ranking (score=0.0). This is correct behaviour but it means the product-level recommendations for customers with very short purchase histories are essentially popularity-based too.

**Cold-start category restriction.** Rule 10 in the rules engine limits new customers to only 3 categories. This is intentionally conservative — without any customer data we don't want to show them a full 8-category catalogue. In a real production system you might use session data (what page they came from, what they clicked on) to expand this.

## 8. Why We Chose A\* Search

### The Problem A\* Solves

Once the rules engine returns a list of eligible categories, we need a way to narrow that list down to the ones most relevant to a specific customer. A returning customer who mainly buys Home Decor items is probably more interested in Kitchen & Dining (which is closely related to Home Decor in the ALS similarity space) than in Stationery & Craft (which is more distantly related). We want the search to respect those relationships.

We modelled this as a graph search problem: the 8 product categories are nodes, and two categories are connected by an edge if their ALS cosine similarity is at least 0.06. The threshold 0.06 was chosen because it's the point where the Fashion & Accessories category becomes fully isolated — its highest similarity to any other category is 0.059, so it's correctly unreachable unless the rules engine explicitly allows it through spend-tier rules.

The search starts at the customer's favourite category and finds all eligible categories reachable within 2 hops. The question then is: in what order should we pass those categories to the ML model?

### Why Not BFS?

BFS was our original approach. It explores the graph hop by hop and returns all reachable categories in order of hop distance. The problem with BFS is that it treats every edge as equal. A category that is 0.19 similar to the start (very close) and a category that is 0.07 similar (barely over the threshold) would both be treated as '1 hop away'. BFS has no way to say 'this neighbour is much more relevant than that one'.

On our specific graph with 8 nodes and ALS-derived edge weights, BFS and A\* actually reach the same *set* of categories. The difference is in the *ordering*. BFS orders by hop count; A\* orders by a combined measure of similarity cost and category popularity. The ML model ultimately re-ranks everything, but giving it a better-ordered shortlist is still a more principled approach.

### Why Not DFS?

DFS goes deep along one path before backtracking. On a small 8-node graph it would find the same nodes eventually, but the order it returns them in is essentially arbitrary — determined by which neighbour happens to be explored first. There's no guarantee of finding the closest or most relevant category before a distant one. DFS doesn't make sense for a problem where we care about ordering by relevance.

### Why A\*?

A\* is informed search — it uses both a path cost (how much it cost to get here) and a heuristic estimate (how far we still are from a good goal state) to decide which node to expand next. This gives it a principled way to order the results that takes both graph structure and category quality into account.

Our A\* formulation:

| Component | Definition | Reasoning |
|---|---|---|
| **State space** | 8 product categories | The nodes of the ALS similarity graph |
| **Initial state** | Customer's `favourite_category` | The most natural starting point |
| **Goal** | Any eligible category within `max_hops` | All eligible reachable nodes are goals |
| **Edge cost g(n)** | `1 − similarity(u, v)` | High similarity = low cost = preferred path |
| **Heuristic h(n)** | `1 − normalised_popularity(n)` | Popular categories = lower h = explored sooner |
| **f(n) = g + h** | Minimised by A\* | Balances proximity to favourite with global desirability |

### Admissibility of the Heuristic

For A\* to find the optimal path, the heuristic must be admissible — it must never overestimate the true cost to reach the goal. Our heuristic h(n) = 1 − normalised\_popularity(n) satisfies this:

- Normalised popularity is computed as `popularity_sum(n) / max_popularity_sum`, so it's always in [0, 1], meaning h(n) is always in [0, 1].
- Every edge cost in our graph is `1 − similarity`, and similarity is always less than 1 (no category is 100% similar to another), so every edge cost is strictly greater than 0.
- This means h(n) ≤ 1 ≤ actual remaining cost. The heuristic never overestimates. A\* is therefore guaranteed to find the minimum-cost ordering.

### What A\* Actually Does for the Recommendation

In practice, A\* produces a shortlist ordered by f-score. Categories that are both close to the customer's favourite (low g) and widely purchased (low h) get lower f-scores and appear first in the shortlist. The RF model then re-scores and re-ranks them, but it receives a shortlist that is already informed by graph structure and popularity — which is a better input than an arbitrarily ordered BFS list.

The search is also deliberately constrained: `max_hops=2` and `threshold=0.06`. These parameters were chosen so Fashion & Accessories (which is nearly isolated in the similarity graph) is not reachable from unrelated categories, and so the search does not wander too far from the customer's known interests.